In [1]:
import json
import ctypes
import struct
import blosc2
import numpy as np

from pymargo.core import Engine
import pyyokan_common as yokan
from pyyokan_client import Client
from pyyokan_server import Provider

In [2]:
def init_engine(protocol, server_addr, provider_id):
    # init engine
    engine = Engine(protocol)
    mid = engine.get_internal_mid()
    addr = engine.lookup(server_addr)
    hg_addr = addr.get_internal_hg_addr()
    provider = Provider(mid=mid, provider_id=provider_id, config='{"database":{"type":"map"}}')
    client = Client(mid=mid)
    db = client.make_database_handle(address=hg_addr, provider_id=provider_id)
    return db

In [3]:
def list_all_keys(db):
    num_keys = db.count()
    
    max_length = 1024
    prefix = ''
    out_keys = []
    for i in range(0, num_keys):
      out_keys.append( bytearray(max_length+len(prefix)+1) )
    
    from_key = ''
    ksizes = db.list_keys(keys=out_keys, from_key=from_key, filter=prefix)
    
    keys = []
    for i in range(len(ksizes)):
        key_size = ksizes[i]
        
        k = out_keys[i]
        key = (k[:key_size]).decode('ascii')
        keys.append(key)
        
    return keys

In [4]:
def split_key(key, pos):
    parts = key.split('/')
    name = parts[pos]
    return name

In [5]:
def get_field_name(key):
    parts = keys[0].split('/')
    name = parts[len(parts)-2]
    return name

In [14]:
def list_fields(db, timestep):
    
    all_keys = list_all_keys(db)
    print(all_keys)
    
    x = []
    for k in all_keys:
        parts = k.split('/')
        name = parts[len(parts)-2]
        x.append(name)
    return list(set(x))

In [7]:
def list_attributes(db, key):
    all_keys = list_all_keys(db)

    x = []
    for k in all_keys:
        parts = k.split('/')
        name = parts[2]
        field = parts[3]
        if name == key:
            x.append(field)
    x = list(set(x))

    return x

In [8]:
def get_value(db, key):
    ''' Get data from the server for that key '''

    # length of the value associated with the key
    l = db.length(key)

    out_val = bytearray(l)          # create buffer
    db.get(key=key, value=out_val)  # get the data
    v = out_val.decode("ascii")     # convert to ascii
    return v

In [9]:
def get_data(db, key):
    ''' Get data from the server for that key '''

    # length of the value associated with the key
    l = db.length(key)

    out_val = bytearray(l)          # create buffer
    db.get(key=key, value=out_val)  # get the data
    return out_val

In [10]:
# Params
provider_id = 123
#protocol = 'na+sm'
protocol = 'ofi+tcp'
#server_addr = 'na+sm://1364645-0'
server_addr = 'ofi+tcp://192.168.100.57:32905'


In [11]:
engine = Engine(protocol)
mid = engine.get_internal_mid()
addr = engine.lookup(server_addr)
hg_addr = addr.get_internal_hg_addr()
provider = Provider(mid=mid, provider_id=provider_id, config='{"database":{"type":"map"}}')
client = Client(mid=mid)
db = client.make_database_handle(address=hg_addr, provider_id=provider_id)

In [12]:
ts = '_2'

In [15]:
keys = list_fields(db, ts)
keys

['_0/0/pressure_2/compressed_size', '_0/0/pressure_2/num_elems', '_0/0/pressure_2/type', '_0/0/pressure_2/value', '_0/0/temperature_2/compressed_size', '_0/0/temperature_2/num_elems', '_0/0/temperature_2/type', '_0/0/temperature_2/value', '_0/1/pressure_2/compressed_size', '_0/1/pressure_2/num_elems', '_0/1/pressure_2/type', '_0/1/pressure_2/value', '_0/1/temperature_2/compressed_size', '_0/1/temperature_2/num_elems', '_0/1/temperature_2/type', '_0/1/temperature_2/value', '_0/2/pressure_2/compressed_size', '_0/2/pressure_2/num_elems', '_0/2/pressure_2/type', '_0/2/pressure_2/value', '_0/2/temperature_2/compressed_size', '_0/2/temperature_2/num_elems', '_0/2/temperature_2/type', '_0/2/temperature_2/value', '_0/3/pressure_2/compressed_size', '_0/3/pressure_2/num_elems', '_0/3/pressure_2/type', '_0/3/pressure_2/value', '_0/3/temperature_2/compressed_size', '_0/3/temperature_2/num_elems', '_0/3/temperature_2/type', '_0/3/temperature_2/value', '_0/4/pressure_2/compressed_size', '_0/4/pressu

['temperature_2', '_2', '_0', 'pressure_2', '_1']

In [ ]:
attributes = list_attributes(db, 'pressure_2')
attributes

In [ ]:
def get_value(db, key, ts):
    # Get all keys
    all_keys = list_all_keys(db)

    # Filter out the keys
    x = []
    for k in all_keys:
        parts = k.split('/')
        name = parts[2]
        field = parts[3]
        ts = parts[0]
        _ts = '_'+str(timestep)

        if _name == name and ts == _ts:
            if field == 'value':
                x.append(k)

    # Get and decompress
    values = []
    for k in x:
        # Get value
        val = get_data(db, k)

        # Get num elems
        num_elem_key = k.replace('value','num_elems')
        num_elems = get_value(db, num_elem_key)
        n_e = str(num_elems) + 'f'

        # decompress
        a_bytesobj2 = blosc2.decompress(val)
        x = struct.unpack(n_e, a_bytesobj2)
        values.append(x)
        
    return values

In [ ]:
_name = 'pressure_2'
timestep = 2

all_keys = list_all_keys(db)

x = []
for k in all_keys:
    parts = k.split('/')
    name = parts[2]
    field = parts[3]
    ts = parts[0]
    _ts = '_'+str(timestep)

    if _name == name and ts == _ts:
        if field == 'value':
            x.append(k)

values = []
for k in x:
    #print(k)
    val = get_data(db, k)
    
    num_elem_key = k.replace('value','num_elems')
    #print(num_elem_key)
    num_elems = get_value(db, num_elem_key)
    n_e = str(num_elems) + 'f'
    #print(num_elms)
    
    a_bytesobj2 = blosc2.decompress(val)
    x = struct.unpack(n_e, a_bytesobj2)
    values.append(x)
    
print(len(values[4]))


In [ ]:
key = 'pressure_2'

def list_attributes(db, key):
    all_keys = list_all_keys(db)

    x = []
    for k in all_keys:
        parts = k.split('/')
        name = parts[2]
        field = parts[3]
        if name == key:
            x.append(field)
    x = list(set(x))

    return x


In [ ]:
get_attributes

In [ ]:
num_keys = db.count()
num_keys

In [ ]:
all_keys = list_all_keys(db)
all_keys

In [ ]:
data_type_key, num_elems_key, compressed_size_key, value_key = get_attributes(all_keys)

In [ ]:
data_type_key, num_elems_key, compressed_size_key, value_key

In [ ]:
ts = '_2'
result = list(filter(lambda x: x.startswith(ts), keys))
print(result)

In [ ]:
result[0]

In [ ]:
split_key(result[0],3)

In [ ]:
name = "pressure_2"
result = list(filter(lambda x: x.count(name), result))
print(result)

In [ ]:
def get_attributes(result):
    data_type_key = ( list(filter(lambda x: x.endswith('type'), result)) )[0]
    num_elems     = ( list(filter(lambda x: x.endswith('num_elems'), result)) )[0]
    comp_size_key = ( list(filter(lambda x: x.endswith('compressed_size'), result)) )[0]
    value_key     = ( list(filter(lambda x: x.endswith('value'), result)) )[0]
    
    return data_type_key, num_elems, comp_size_key, value_key

In [ ]:
def get_value(db, key):
    ''' Get data from the server for that key '''

    # length of the value associated with the key
    l = db.length(key)

    out_val = bytearray(l)          # create buffer
    db.get(key=key, value=out_val)  # get the data
    v = out_val.decode("ascii")     # convert to ascii
    return v

In [ ]:
def get_data(db, key):
    ''' Get data from the server for that key '''

    # length of the value associated with the key
    l = db.length(key)

    out_val = bytearray(l)          # create buffer
    db.get(key=key, value=out_val)  # get the data
    return out_val

In [ ]:
data_type_key, num_elems_key, compressed_size_key, value_key = get_attributes(result)

In [ ]:
data_type_key, num_elems_key, compressed_size_key, value_key

In [ ]:
data_type_key = ( list(filter(lambda x: x.endswith('num_elems'),      result)) )[0]
data_type_key

In [ ]:
name = get_field_name(keys[0])
name

In [ ]:
get_value(db, data_type_key)

In [ ]:
get_value(db, num_elems_key)

In [ ]:
get_value(db, compressed_size_key)

In [ ]:
def get_attributes(result):
    data_type_key = ( list(filter(lambda x: x.endswith('type'),      result)) )[0]
    num_elems     = ( list(filter(lambda x: x.endswith('num_elems'), result)) )[0]
    comp_size_key     = ( list(filter(lambda x: x.endswith('compressed_size'), result)) )[0]
    value_key     = ( list(filter(lambda x: x.endswith('value'), result)) )[0]
    
    return split_key(data_type_key,3), split_key(num_elems,3), split_key(comp_size_key,3), split_key(value_key,3)

In [ ]:
data_type_key = ( list(filter(lambda x: x.endswith('type'),      result)) )[0]
num_elems     = ( list(filter(lambda x: x.endswith('num_elems'), result)) )[0]
comp_size_key     = ( list(filter(lambda x: x.endswith('compressed_size'), result)) )[0]
value_key     = ( list(filter(lambda x: x.endswith('value'), result)) )[0]

In [ ]:
data_type_key

In [ ]:
comp_size_key

In [ ]:
value_key

In [ ]:
name = get_field_name(keys[0])
print(name)

In [ ]:
get_data(db, data_type_key)

In [ ]:
get_data(db, num_elems)

In [ ]:
get_data(db, comp_size_key)

In [ ]:
value_key

In [ ]:
def get_data(db, key):
    ''' Get data from the server for that key '''

    # length of the value associated with the key
    l = db.length(key)

    out_val = bytearray(l)          # create buffer
    db.get(key=key, value=out_val)  # get the data
    return out_val

In [ ]:
buffer = get_data(db, value_key)

In [ ]:
type(buffer)

In [ ]:
len(buffer)

In [ ]:
a_bytesobj2 = blosc2.decompress(buffer)

In [ ]:
type(a_bytesobj2)

In [ ]:
len(a_bytesobj2)

In [ ]:
a_bytesobj2

In [ ]:
x = struct.unpack('50f', a_bytesobj2)

In [ ]:
a_bytesobj2 = blosc2.decompress(buffer)
x = struct.unpack('50f', a_bytesobj2)

In [ ]:
x

In [ ]:
type(x[0])

In [ ]:
help(blosc2.decompress)

In [ ]:
c_bytesobj = blosc2.decompress(a_bytesobj, typesize=4)

In [ ]:
def get_value(db, field, ts):
    
    # Get the list of keys
    num_keys = db.count()
    keys = list_keys(db, num_keys)
    print(keys)
    
    # Filter by name and timestep
    result_1 = list( filter(lambda x: x.startswith( '_' + str(ts) ), keys) )
    result_2 = list( filter(lambda x: x.count(field)  , result_1) )
    
    data_type_key = ( list(filter(lambda x: x.endswith('type'),      result_2)) )[0]
    num_elems_key = ( list(filter(lambda x: x.endswith('num_elems'), result_2)) )[0]
    query_key     = ( list(filter(lambda x: x.endswith('value'), result_2)) )[0]
    rank_key      = ( list(filter(lambda x: x.endswith('rank'), result_2)) )[0]
    print("num_ranks",rank_key)
    
    str_data  = get_data(db, query_key)
    num_elems = get_data(db, num_elems_key)
    data_type = get_data(db, data_type_key)
    ranks = get_data(db, rank_key)
    print("ranks",ranks)
    
    data = deserialize(str_data, int(num_elems), data_type)
    return data

In [ ]:
data = get_value(db,'energy',4)
data

In [ ]:
get_data(db, key)